# Whisper base 전사 — 누시 파일럿

wav 파일을 Whisper `base` 모델로 전사하고, 결과를 txt / json / srt / eaf 로 저장합니다.
언어 토큰은 지정하지 않고 모델이 스스로 판정하게 둡니다.

**실행 순서** — 위에서부터 셀을 차례로 실행하세요. 셀 1~3은 한 번만 실행하면 됩니다.


## 1. 런타임 확인

GPU가 붙어 있는지 봅니다. `base` 모델은 CPU로도 돌아가지만 GPU가 있으면 훨씬 빠릅니다.

GPU가 없다고 나오면 상단 메뉴에서 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU**
를 선택하고 다시 실행하세요.


In [ ]:
import subprocess, sys

try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True, check=True)
    print("GPU:", out.stdout.strip())
    DEVICE = "cuda"
except Exception:
    print("GPU 없음 — CPU로 실행합니다 (base 모델이면 충분합니다)")
    DEVICE = "cpu"

print("Python:", sys.version.split()[0])
print("device =", DEVICE)

## 2. 설치

`openai-whisper` 와 오디오 디코딩용 `ffmpeg` 를 설치합니다.
2~3분 걸립니다. 설치 후 런타임 재시작은 필요 없습니다.


In [ ]:
!pip -q install -U openai-whisper
!apt-get -qq install -y ffmpeg > /dev/null

import whisper
print("whisper", whisper.__version__ if hasattr(whisper, "__version__") else "(ok)")

## 3. 오디오 업로드

두 가지 방법이 있습니다. 하나만 실행하세요.

**A. 직접 업로드** — 파일이 작을 때 (수십 MB 이하). 아래 셀을 실행하면 파일 선택창이 뜹니다.

**B. Google Drive 연결** — 파일이 크거나 반복 작업할 때. 다음 셀을 쓰세요.


In [ ]:
# A. 직접 업로드
from google.colab import files
from pathlib import Path

uploaded = files.upload()
AUDIO = Path(next(iter(uploaded)))
print("업로드됨:", AUDIO, f"({AUDIO.stat().st_size/1e6:.1f} MB)")

In [ ]:
# B. Google Drive 연결 (A 대신 쓸 경우에만 실행)
# from google.colab import drive
# from pathlib import Path
# drive.mount("/content/drive")
# AUDIO = Path("/content/drive/MyDrive/nouchi/audio.wav")   # 경로를 수정하세요
# assert AUDIO.exists(), f"파일 없음: {AUDIO}"
# print("경로:", AUDIO)

## 4. 오디오 확인

전사 전에 파일이 제대로 읽히는지, 길이와 샘플레이트가 어떤지 봅니다.

Whisper는 내부적으로 **16 kHz 모노**로 리샘플링하므로 원본이 다른 포맷이어도 상관없습니다.
다만 원본 스펙을 기록해두면 나중에 논문의 자료 기술 절에 씁니다.


In [ ]:
import wave, contextlib, whisper

try:
    with contextlib.closing(wave.open(str(AUDIO), "rb")) as w:
        print(f"채널   : {w.getnchannels()}")
        print(f"샘플레이트: {w.getframerate()} Hz")
        print(f"비트깊이 : {w.getsampwidth()*8} bit")
        print(f"길이   : {w.getnframes()/w.getframerate():.2f} 초")
except Exception as e:
    print("wave 모듈로 못 읽음 (wav가 아니거나 압축 포맷):", e)

audio = whisper.load_audio(str(AUDIO))     # 16kHz float32 로 변환
DURATION = len(audio) / 16000
print(f"\nWhisper 로드 완료: {DURATION:.2f} 초")

## 5. 모델 로드

`base` 모델을 불러옵니다. 다국어 모델이며 약 74M 파라미터, 다운로드 약 140 MB입니다.

`base.en` 이 아니라 `base` 여야 합니다. `.en` 붙은 모델은 영어 전용이라 언어 판정 자체를
하지 않습니다.


In [ ]:
import whisper

MODEL_NAME = "base"
model = whisper.load_model(MODEL_NAME, device=DEVICE)

print(f"모델: {MODEL_NAME}")
print(f"다국어 지원: {model.is_multilingual}")
print(f"언어 토큰 수: {model.num_languages if hasattr(model, 'num_languages') else 99}")

## 6. 언어 자동 판정

Whisper의 기본 자동 감지는 **첫 30초만 보고** 전체 언어를 결정합니다.
담화 중간에 다른 언어가 섞이거나 도입부 음향이 나쁘면 판정이 틀어질 수 있습니다.

그래서 여러 구간을 표본으로 뽑아 각각 판정해봅니다. 이 결과는 전사에 강제로 넣지 않고
**기록용**입니다. 모델이 이 발화를 무엇으로 보는지, 그 판단이 구간마다 흔들리는지를
관찰하는 것이 목적입니다.

상위 5개 후보와 확률이 함께 출력됩니다.


In [ ]:
import numpy as np, whisper

N_WINDOWS = 5          # 표본 창 개수
WIN = 30 * 16000       # Whisper 입력 단위 = 30초

total = len(audio)
if total <= WIN:
    starts = [0]
else:
    step = max(1, (total - WIN) // max(1, N_WINDOWS - 1))
    starts = [min(i * step, total - WIN) for i in range(N_WINDOWS)]

detections = []
for st in starts:
    chunk = whisper.pad_or_trim(audio[st:st + WIN])
    mel = whisper.log_mel_spectrogram(chunk, n_mels=model.dims.n_mels).to(model.device)
    _, probs = model.detect_language(mel)
    if isinstance(probs, list):
        probs = probs[0]
    ranked = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)[:5]
    detections.append({"start_s": st/16000,
                       "end_s": min(st+WIN, total)/16000,
                       "top5": [(l, float(p)) for l, p in ranked]})
    top = ", ".join(f"{l}:{p:.3f}" for l, p in ranked)
    print(f"{st/16000:7.1f}s ~ {min(st+WIN,total)/16000:7.1f}s   {top}")

langs = [d["top5"][0][0] for d in detections]
print(f"\n창별 1순위: {langs}")
print("판정 일치" if len(set(langs)) == 1 else "판정 불일치 — 구간마다 다르게 봅니다")

## 7. 전사

`language=None` 으로 넘겨 Whisper 내부의 자동 감지에 그대로 맡깁니다.
위에서 얻은 판정 결과를 강제로 주입하지 않습니다 — 기본 동작을 관찰하는 것이 목적이니까요.

**설정 설명**

- `temperature=0.0` — 탐욕적 디코딩. 매번 같은 결과가 나와 재현성이 확보됩니다.
  기본값은 실패 시 온도를 올리며 재시도하는데, 그러면 실행마다 결과가 달라집니다.
- `word_timestamps=True` — 단어 단위 시각. 강제정렬 없이도 구간별 분석이 가능해집니다.
- `condition_on_previous_text` — 이전 텍스트를 다음 디코딩의 조건으로 넣을지.
  켜두면 문맥 일관성이 좋아지지만 **반복 루프 환각**이 잘 생깁니다.
  같은 문장이 계속 반복되는 출력이 나오면 이 값을 `False` 로 바꿔 다시 돌려보세요.
- `initial_prompt` — 디코더의 사전확률을 밀어주는 프롬프트. 지금은 비워둡니다.
  나중에 누시 어휘 목록을 넣어 개선 여부를 보는 실험에 씁니다.


In [ ]:
import time

t0 = time.time()
result = model.transcribe(
    audio,
    language=None,                    # 자동 감지
    task="transcribe",
    temperature=0.0,
    word_timestamps=True,
    condition_on_previous_text=True,  # 반복이 심하면 False 로
    initial_prompt=None,
    verbose=False,
)
elapsed = time.time() - t0

print(f"전사 완료: {elapsed:.1f}초 (오디오 {DURATION:.1f}초, 실시간 대비 {DURATION/elapsed:.1f}x)")
print(f"전사에 사용된 언어: {result['language']}")
print(f"세그먼트 수: {len(result['segments'])}")

## 8. 결과 확인

세그먼트별로 시각과 텍스트를 봅니다. 오른쪽 세 열은 Whisper가 스스로 뱉는 지표입니다.

- `logprob` — 평균 로그확률. 낮을수록(음수 쪽으로 클수록) 모델이 자신 없어 한 구간입니다.
- `comp` — 압축비. 값이 크면 같은 표현이 반복되고 있다는 뜻이라 **반복 환각의 신호**입니다.
  2.4를 넘으면 Whisper 자체가 실패로 간주하는 임계값입니다.
- `nosp` — 무음일 확률. 높은데 텍스트가 있으면 **없는 말을 지어냈을** 가능성이 있습니다.

이 세 지표가 나쁜 구간을 먼저 청취하면 효율이 좋습니다.


In [ ]:
def ts(t):
    return f"{int(t//60):02d}:{t%60:06.3f}"

print(f"{'구간':<22}{'logprob':>9}{'comp':>7}{'nosp':>7}  텍스트")
print("-" * 100)
for s in result["segments"]:
    print(f"[{ts(s['start'])} → {ts(s['end'])}]"
          f"{s.get('avg_logprob', 0):>9.3f}"
          f"{s.get('compression_ratio', 0):>7.2f}"
          f"{s.get('no_speech_prob', 0):>7.3f}  {s['text'].strip()}")

### 주의가 필요한 구간만 추리기

임계값을 넘는 구간을 따로 뽑습니다. 청취 우선순위를 정하는 데 씁니다.


In [ ]:
flagged = []
for i, s in enumerate(result["segments"]):
    reasons = []
    if s.get("avg_logprob", 0) < -1.0:      reasons.append("낮은 확률")
    if s.get("compression_ratio", 0) > 2.4: reasons.append("반복 의심")
    if s.get("no_speech_prob", 0) > 0.5:    reasons.append("무음 가능성")
    if reasons:
        flagged.append((i, s, reasons))

print(f"플래그된 구간: {len(flagged)} / {len(result['segments'])}\n")
for i, s, r in flagged:
    print(f"#{i:3d} [{ts(s['start'])}] {', '.join(r):<24} {s['text'].strip()[:60]}")

## 9. 저장 — txt / json / srt

- **txt** — 전체 텍스트만. 빠르게 읽어볼 때.
- **json** — 세그먼트·단어 시각과 모델 지표까지 전부. **정량 분석은 이 파일로 합니다.**
- **srt** — 자막 형식. 영상에 얹어 확인할 때.


In [ ]:
import json
from pathlib import Path

STEM = AUDIO.stem
OUT = Path("output"); OUT.mkdir(exist_ok=True)

# txt
(OUT / f"{STEM}.{MODEL_NAME}.txt").write_text(result["text"].strip(), encoding="utf-8")

# json (언어 판정 기록 포함)
payload = {
    "audio": str(AUDIO),
    "model": MODEL_NAME,
    "duration_s": DURATION,
    "transcription_language": result["language"],
    "window_detections": detections,
    "segments": [
        {"id": i, "start": s["start"], "end": s["end"], "text": s["text"],
         "avg_logprob": s.get("avg_logprob"),
         "compression_ratio": s.get("compression_ratio"),
         "no_speech_prob": s.get("no_speech_prob"),
         "words": [{"word": w["word"], "start": w["start"], "end": w["end"],
                    "probability": w.get("probability")}
                   for w in (s.get("words") or [])]}
        for i, s in enumerate(result["segments"])
    ],
}
(OUT / f"{STEM}.{MODEL_NAME}.json").write_text(
    json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

# srt
def srt_time(t):
    h, r = divmod(t, 3600); m, s = divmod(r, 60)
    return f"{int(h):02d}:{int(m):02d}:{int(s):02d},{int((s%1)*1000):03d}"

lines = []
for i, s in enumerate(result["segments"], 1):
    lines += [str(i), f"{srt_time(s['start'])} --> {srt_time(s['end'])}",
              s["text"].strip(), ""]
(OUT / f"{STEM}.{MODEL_NAME}.srt").write_text("\n".join(lines), encoding="utf-8")

for f in sorted(OUT.iterdir()):
    print(f"{f.name:<40}{f.stat().st_size:>10,} bytes")

## 10. 저장 — ELAN(.eaf)

ELAN에서 열 수 있는 형식으로 저장합니다. 티어 네 개를 만듭니다.

| 티어 | 내용 |
|---|---|
| `transcription` | 세그먼트 단위 전사 |
| `words` | 단어 단위 |
| `language` | 구간별 언어 판정과 확률 |
| `notes` | 수기 주석용 빈 티어 |

**구간 정리가 필요한 이유** — ELAN은 같은 티어 안에서 주석이 겹치거나 길이가 0인 것을
허용하지 않습니다. Whisper 출력에는 경계가 맞물리거나 길이 0인 구간이 섞이므로
`sanitize()` 가 이를 보정합니다. 보정된 구간은 원래 위치에서 밀릴 수 있으니,
정밀한 시각이 필요하면 json 쪽 원본을 참조하세요.


In [ ]:
import xml.etree.ElementTree as ET
from xml.dom import minidom
from datetime import datetime, timezone

def sanitize(spans, min_ms=10):
    out, prev_end = [], 0
    for st, en, tx in sorted(spans, key=lambda x: (x[0], x[1])):
        tx = (tx or "").strip()
        if not tx:
            continue
        st = max(int(st), prev_end)
        en = max(int(en), st + min_ms)
        out.append((st, en, tx)); prev_end = en
    return out

def build_eaf(tiers, media: Path = None, author="colab-whisper"):
    slots, order, tier_rows, ann = {}, [], [], [0]
    def slot(ms):
        ms = int(ms)
        if ms not in slots:
            sid = f"ts{len(slots)+1}"; slots[ms] = sid; order.append((sid, ms))
        return slots[ms]
    for name, spans in tiers:
        rows = []
        for st, en, tx in spans:
            ann[0] += 1
            rows.append((f"a{ann[0]}", slot(st), slot(en), tx))
        tier_rows.append((name, rows))

    root = ET.Element("ANNOTATION_DOCUMENT", {
        "AUTHOR": author,
        "DATE": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "FORMAT": "3.0", "VERSION": "3.0",
        "xmlns:xsi": "http://www.w3.org/2001/XMLSchema-instance",
        "xsi:noNamespaceSchemaLocation": "http://www.mpi.nl/tools/elan/EAFv3.0.xsd"})
    hdr = ET.SubElement(root, "HEADER", {"MEDIA_FILE": "", "TIME_UNITS": "milliseconds"})
    if media is not None:
        ET.SubElement(hdr, "MEDIA_DESCRIPTOR", {
            "MEDIA_URL": media.resolve().as_uri(),
            "RELATIVE_MEDIA_URL": "./" + media.name,
            "MIME_TYPE": "audio/x-wav"})
    ET.SubElement(hdr, "PROPERTY", {"NAME": "lastUsedAnnotationId"}).text = str(ann[0])

    to = ET.SubElement(root, "TIME_ORDER")
    for sid, ms in sorted(order, key=lambda x: x[1]):
        ET.SubElement(to, "TIME_SLOT", {"TIME_SLOT_ID": sid, "TIME_VALUE": str(ms)})

    for name, rows in tier_rows:
        t = ET.SubElement(root, "TIER",
                          {"LINGUISTIC_TYPE_REF": "default-lt", "TIER_ID": name})
        for aid, s1, s2, tx in rows:
            a = ET.SubElement(t, "ANNOTATION")
            al = ET.SubElement(a, "ALIGNABLE_ANNOTATION", {
                "ANNOTATION_ID": aid, "TIME_SLOT_REF1": s1, "TIME_SLOT_REF2": s2})
            ET.SubElement(al, "ANNOTATION_VALUE").text = tx

    ET.SubElement(root, "LINGUISTIC_TYPE", {
        "GRAPHIC_REFERENCES": "false", "LINGUISTIC_TYPE_ID": "default-lt",
        "TIME_ALIGNABLE": "true"})
    ET.SubElement(root, "LOCALE", {"LANGUAGE_CODE": "en"})
    for stereo, desc in [
        ("Time_Subdivision", "Time subdivision of parent annotation's time interval, no time gaps allowed within this interval"),
        ("Symbolic_Subdivision", "Symbolic subdivision of a parent annotation. Annotations refering to the same parent are ordered"),
        ("Symbolic_Association", "1-1 association with a parent annotation"),
        ("Included_In", "Time alignable annotations within the parent annotation's time interval, gaps are allowed")]:
        ET.SubElement(root, "CONSTRAINT", {"DESCRIPTION": desc, "STEREOTYPE": stereo})
    return root

seg_spans = sanitize([(s["start"]*1000, s["end"]*1000, s["text"])
                      for s in result["segments"]])
word_spans = sanitize([(w["start"]*1000, w["end"]*1000, w["word"])
                       for s in result["segments"] for w in (s.get("words") or [])],
                      min_ms=1)
lang_spans = sanitize([(d["start_s"]*1000, d["end_s"]*1000,
                        " | ".join(f"{l}:{p:.2f}" for l, p in d["top5"][:3]))
                       for d in detections])

root = build_eaf([("transcription", seg_spans), ("words", word_spans),
                  ("language", lang_spans), ("notes", [])], media=AUDIO)
eaf_path = OUT / f"{STEM}.{MODEL_NAME}.eaf"
eaf_path.write_bytes(
    minidom.parseString(ET.tostring(root, encoding="utf-8"))
           .toprettyxml(indent="    ", encoding="UTF-8"))

print(f"저장: {eaf_path}")
print(f"  transcription {len(seg_spans)}  words {len(word_spans)}  language {len(lang_spans)}")
print("\nELAN에서 열 때 wav 파일을 같은 폴더에 두세요.")

## 11. 내려받기

결과 폴더를 zip으로 묶어 내려받습니다.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive(f"{STEM}_{MODEL_NAME}", "zip", OUT)
files.download(f"{STEM}_{MODEL_NAME}.zip")

## 다음 단계

이 노트북은 `base` 하나만 돌립니다. 비교 실험을 하려면 셀 5의 `MODEL_NAME` 을
`medium`, `large-v3` 으로 바꿔 각각 실행하고 결과 json을 모으세요.
모델을 바꿀 때는 메모리 확보를 위해 다음을 먼저 실행하는 것이 좋습니다.

```python
del model
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
```

**통제할 것** — 모델만 바꾸고 나머지 설정(temperature, condition_on_previous_text,
initial_prompt)은 고정해야 차이의 원인을 모델로 귀속시킬 수 있습니다.

**주의** — `base` 와 `large-v3` 은 파라미터 수만 다른 것이 아니라 학습 데이터 규모와
시점도 다릅니다. 결과를 "크기 효과"로 단정하지 말고 "모델 계열 간 차이"로 서술하는 편이
정확합니다.
